# Оценка релевантности организаций запросам на Яндекс.Картах с помощью LLM-агента

<img src="https://sun9-65.userapi.com/impg/N4y2cxlL7PauAs82tBNFOUAiNctFICWDy4Mbiw/Jiz1fb7NLWU.jpg?size=1080x1080&quality=95&sign=df2786058624d9ccac3ede4d5d056e2f&type=album" width="500" height="500" />


## Описание и загрузка данных

Данные: https://disk.yandex.ru/d/6d5hFHvpAZjQdw

Ваша задача -- предсказать колонку relevance, используя все остальные данные об организации. Загрузим данные и посмотрим на них

In [ ]:
import requests

public_url = "https://disk.yandex.ru/d/6d5hFHvpAZjQdw"  # твоя публичная ссылка
api_url = "https://cloud-api.yandex.net/v1/disk/public/resources/download"

resp = requests.get(api_url, params={"public_key": public_url})
resp.raise_for_status()
download_url = resp.json()["href"]  # это уже прямая ссылка на файл

dest = "/content/data.jsonl"
with requests.get(download_url, stream=True) as r:
    r.raise_for_status()
    with open(dest, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)


In [ ]:
import json
import pandas as pd

records = []
with open("/content/data.jsonl", encoding="utf-8") as f:
    for i, line in enumerate(f, start=1):
        if i == 2659:      # пропускаем битую строку
            continue
        try:
            obj = json.loads(line)
            records.append(obj)
        except Exception as e:
            print("ещё битая строка:", i, e)

data = pd.DataFrame(records)


In [ ]:
data['relevance'].unique()

array([1. , 0. , 0.1])

In [ ]:
data['relevance'].value_counts()

,count
relevance,
1.0,15881
0.0,14509
0.1,4703


Здесь 1.0 соответствует оценке RELEVANT_PLUS, 0.1 -- оценке RELEVANT_MINUS, 0.0 -- оценке IRRELEVANT.

Ваша задача -- построить LLM-агента, который будет предсказывать релевантность.

Выделим данные для оценки качества агента. Запуск агента -- это тяжелая и потенциально дорогая операция. Поэтому eval-множество имеет размер 500. Также для простоты из eval-множества выкинуты данные с оценкой RELEVANT_MINUS. Тем не менее, вы можете использовать такие примеры для подачи примеров агенту.

**ОБРАТИТЕ ВНИМАНИЕ, ЧТО В EVAL-ДАННЫЕ НЕЛЬЗЯ ПОДГЛЯДЫВАТЬ ДЛЯ КАЛИБРОВКИ АГЕНТА!!! ДЛЯ ЭТОГО ЕСТЬ ОБУЧАЮЩИЕ ДАННЫЕ**

В качестве метрики качества мы будем использовать обычную ACCURACY, поскольку классы сбалансированы.

In [ ]:
train_data = data[570:]
eval_data = data[:570]
eval_data = eval_data[eval_data["relevance"] != 0.1]
eval_data

,Text,address,name,normalized_main_rubric_name_ru,permalink,prices_summarized,relevance,reviews_summarized
0,сигары,"Москва, Дубравная улица, 34/29",Tabaccos; Магазин Tabaccos; Табаккос,Магазин табака и курительных принадлежностей,1263329400,None,1.0,"Организация занимается продажей табака, курите..."
1,кальянная спб мероприятия,"Санкт-Петербург, Большой проспект Петроградско...",PioNero; Pionero; Пицца Паста бар; Pio Nero; P...,Кафе,228111266197,PioNero предлагает разнообразные блюда итальян...,0.0,"Организация PioNero — это кафе, бар и ресторан..."
2,Эпиляция,"Московская область, Одинцово, улица Маршала Жу...",MaxiLife; Центр красоты и здоровья MaxiLife; Ц...,Стоматологическая клиника,1247255817,"Стоматологическая клиника, массажный салон и к...",1.0,"Организация занимается стоматологическими, кос..."
4,стиральных машин,"Москва, улица Обручева, 34/63",М.Видео; M Video; M. Видео; M.Видео; Mvideo; М...,Магазин бытовой техники,1074529324,М.Видео предлагает широкий ассортимент бытовой...,1.0,Организация занимается продажей бытовой техник...
5,сеть быстрого питания,"Санкт-Петербург, 1-я Красноармейская улица, 15",Rostic's; KFC; Ресторан быстрого питания KFC,Быстрое питание,1219173871,Rostic's предлагает различные наборы быстрого ...,1.0,"Организация занимается быстрым питанием, предо..."
...,...,...,...,...,...,...,...,...
561,наращивание ресниц,"Саратов, улица имени А.С. Пушкина, 1",Сила; Sila; Beauty brow; Студия бровей Beauty ...,Салон красоты,236976975812,Салон красоты «Сила» предлагает услуги по уход...,1.0,Организация «Сила» занимается предоставлением ...
565,игры,"Москва, Щёлковское шоссе, 79, корп. 1",YouPlay; YouPlay КиберКлуб,Компьютерный клуб,109673025161,YouPlay КиберКлуб предлагает услуги по игре на...,0.0,Организация занимается предоставлением услуг к...
566,домашний интернет в курске что подключить отзы...,"Курск, Садовая улица, 5",Цифровой канал; Digital Channel; DChannel; ЦК;...,Телекоммуникационная компания,1737991898,None,0.0,None
567,гостиница волгодонск сауна номер телефона,"Ростовская область, городской округ Волгодонск...",Поплавок; Poplavok,"База , дом отдыха",147783493467,"Предлагает размещение в различных типах жилья,...",0.0,Организация «Поплавок» предлагает услуги базы ...


In [ ]:
eval_data.to_excel("eval_data.xlsx")

## Импорты

In [ ]:
!pip install -q langchain langgraph langchain_openai langchain_core langchain_community

In [ ]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END, MessagesState
from langgraph.prebuilt import ToolNode
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_community.tools import TavilySearchResults
from langchain_core.tools import tool

import json
import re
from typing import TypedDict, Dict, Any, List, Optional, Literal
import os
import time, uuid
import math
from dotenv import load_dotenv

In [ ]:
load_dotenv()

True

# Агент


In [ ]:
SIMPLE_PRE_PROMPT = """
Ты — строгий классификатор релевантности организации рубричному запросу.

ВАЖНО:
- По умолчанию label = 0.
- label = 1 ТОЛЬКО если есть подтверждение в карточке организации.
- Если в карточке есть ЯВНОЕ противоречие запросу — label = 0.
- Если явного подтверждения нет и явного противоречия нет, то needs_search = true.

Ты НЕ должен додумывать. Нет доказательства => needs_search=true.

Формат ответа (строго JSON):
{
  "label": 0 or 1,
  "needs_search": true or false,
  "rationale": "1 предложение",
  "evidence": ["короткие цитаты/фрагменты из карточки, если были"]
}

Вход:
- query: рубричный запрос пользователя
- org: {name, address, rubric}
- card_text: текст карточки организации

Верни только JSON.
""".strip()


In [ ]:
SIMPLE_POST_PROMPT = """
Ты — строгий классификатор релевантности организации рубричному запросу.
То есть тебе дана информация об организации и запрос, тебе нужно определить,
релевантен ли запрос организации (label = 1), или не релевантен (label=0)
ВАЖНО:
- По умолчанию label = 0.
- Если card_text или internet_text ЯВНО подтверждают релевантность запроса, то label = 1,
- субъективные прилагательные в запросе сверяй с отзывами
(например, если несколько отзывов оценивают место как романтичное, то оно считается романтичным)
- internet_text — это просто фрагменты текста из интернета, он может быть нерелевантен.
Используй его только если там явно указаны: организация (по названию/адресу) И требуемая деталь.
- В блоке internet_text, текст из разных источников разделен символом <NEXT_SOURCE>,
тексты, разделенные <NEXT_SOURCE> никак не связаны.
- Если запрос сторого релевантен организации, то label = 1.
- Если запрос нельзя назвать сторо релевантным, то label = 0.

Формат ответа (строго JSON):
{
  "label": 0 or 1,
  "rationale": "1 предложение",
  "evidence": ["короткие цитаты/фрагменты из card_text/internet_text, если были"]
}

Вход:
- query
- org: {name, address, rubric}
- card_text
- internet_text (до 3000 символов)

Верни только JSON.
""".strip()


In [ ]:
# OpenRouter LLM
open_router_model_name = 'arcee-ai/trinity-large-preview:free'

llm = ChatOpenAI(
    model=open_router_model_name,
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
    temperature=0.1,
    max_tokens=8500,
    timeout=60,
    max_retries=3,
)

tavily = TavilySearchResults(
    max_results=10,
    search_depth="advanced"
)


/tmp/ipython-input-2665307303.py:14: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavily = TavilySearchResults(


In [ ]:
p = 'Привет! Какая ты модель? '
ans = llm.invoke([SystemMessage(p)])

NotFoundError: Error code: 404 - {'error': {'message': 'The free MiMo-V2-Flash period has ended. To continue using this model, please migrate to the paid slug: xiaomi/mimo-v2-flash', 'code': 404}, 'user_id': 'user_35lMavUOjIClZzhxsoLbmYABpHr'}

In [ ]:
def _extract_json(text: str) -> str:
    text = (text or "").strip()
    if text.startswith("{") and text.endswith("}"):
        return text
    m = re.search(r"\{.*\}", text, flags=re.DOTALL)
    return m.group(0) if m else ""

def llm_json(system_prompt: str, payload: Dict[str, Any], repair: int = 1) -> Dict[str, Any]:
    """Задача: безопасно получить dict из LLM, даже если она с первого раза вернула невалидный JSON """
    msgs = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=json.dumps(payload, ensure_ascii=False))
    ]
    resp = llm.invoke(msgs)
    raw = resp.content if isinstance(resp.content, str) else str(resp.content)
    cand = _extract_json(raw)

    try:
        if not cand:
            raise json.JSONDecodeError("Empty JSON", raw, 0)
        return json.loads(cand)
    except Exception:
        if repair <= 0:
            raise

    msgs.append(HumanMessage(content="Верни только валидный JSON. Никакого текста вне JSON."))
    resp2 = llm.invoke(msgs)
    raw2 = resp2.content if isinstance(resp2.content, str) else str(resp2.content)
    cand2 = _extract_json(raw2)
    if not cand2:
        raise json.JSONDecodeError("Empty JSON after repair", raw2, 0)
    return json.loads(cand2)


In [ ]:
def build_search_query_simple(org: Dict[str, Any], query: str) -> str:
    """
    Создает простой веб-запрос:
    "<query> <organization_name> <address> <query>"
    """
    q = (query or "").strip()
    name = max(str(org.get("name", "") or "").split(';'), key = len).strip()
    addr = str(org.get("address", "") or "")

    parts = [p for p in [q, name, addr, q] if p]
    out = " ".join(parts)
    return re.sub(r"\s+", " ", out)[:320]


In [ ]:
def tavily_search_blob(search_query: str, max_chars: int = 3000) -> str:
    res = tavily.invoke({"query": search_query})
    if isinstance(res, dict):
        res = [res]

    chunks = []
    total = 0
    for item in (res or []):
        if type(item) != dict:
            continue
        title = (item.get("title") or "").strip()
        url = (item.get("url") or "").strip()
        content = (item.get("content") or item.get("snippet") or "").strip()
        if not (title or content):
            continue

        block = f"Title: {title}\nText: {content}".strip()
        block = block[:450]  # чтобы влезло больше источников

        add_len = len(block) + 2
        if total + add_len > max_chars:
            break

        chunks.append(block)
        total += add_len

    return "\n\n<NEXT_SOURCE> ".join(chunks).strip()


In [ ]:
org = train_data.iloc[123]
query = org['Text']

org_min = {
        "name": _clean_str(org.get("name", "")),
        "address": _clean_str(org.get("address", "")),
        "rubric": _clean_str(org.get("normalized_main_rubric_name_ru", org.get("rubric", ""))),
}

search_q = build_search_query_simple(org_min, query)
internet_text = tavily_search_blob(search_q, max_chars=2000)

In [ ]:
train_data.iloc[123]

,693
Text,"танцы детские серпухов с 1,5 лет"
address,"Московская область, Серпухов, Московское шоссе..."
name,Выше Крыш; Vyshe Krysh
normalized_main_rubric_name_ru,Центр развития ребёнка
permalink,40806652631
prices_summarized,Центр развития ребёнка «Выше Крыш» предлагает ...
relevance,0.0
reviews_summarized,Организация «Выше Крыш» занимается развитием д...


In [ ]:
print(internet_text)

Title: Выше Крыш, центр развития ребёнка, Московское ш., 49, Серпухов
Text: Центр развития ребёнка «Выше Крыш» — «Хорошее место» по оценкам пользователей — по адресу Московская область, Серпухов, Московское шоссе, 49,

<NEXT_SOURCE> Title: Центр развития детей «Выше Крыш» г.Серпухов - ВКонтакте
Text: Описание: https://vk.com/app6013442_-39015327?form_id=12#form_i.. · Телефон: +7 (925) 086-86-46 · Место: Московское шоссе 49, Серпухов.

<NEXT_SOURCE> Title: "Dance Kids" - уроки танцев в Серпухове
Text: Индекс: 142210 · Полный адрес: Московская область, Серпухов, Большой Ударный переулок, 1 · Все телефоны компании: +7 (900) 059-08-08 · Социальные сети:

<NEXT_SOURCE> Title: Центр развития детей ВЫШЕ КРЫШ г.Серпухов - занятия для ...
Text: Яркие зимние каникулы для детей 6-10 лет! ❄️✨    Путешествие в будущее с 17.02 - 21.02 🎉   Приглаша…
Дорогие друзья! 🌟    Напоминаем вам о Неделе открытых дверей в нашем центре Выше крыш!    Для тех…
 Дорогие друзья! 🌟    Напоминаем вам о Неделе открытых

In [ ]:
def simple_agent_predict(row: Dict[str, Any]) -> Dict[str, Any]:
    query = _clean_str(row.get("Text", ""))

    org = row  # row dict
    org_min = {
        "name": _clean_str(org.get("name", "")),
        "address": _clean_str(org.get("address", "")),
        "rubric": _clean_str(org.get("normalized_main_rubric_name_ru", org.get("rubric", ""))),
    }

    card_text = make_card_text(row)

    # ---- PRE ----
    pre_payload = {"query": query, "org": org_min, "card_text": card_text}
    pre_raw = llm_json(SIMPLE_PRE_PROMPT, pre_payload, repair=1)
    pre = post_validate_pre(pre_raw)

    out = {
        "label": pre["label"],
        "used_search": False,
        "search_query": "",
        "internet_text": "",
        "rationale": pre["rationale"],
        "evidence": pre["evidence"],
    }

    # если карточка уже явно подтверждает
    if pre["label"] == 1 and not pre["needs_search"]:
        return out

    # если поиск не нужен (явное противоречие) — оставляем 0
    if not pre["needs_search"]:
        out["label"] = 0
        return out

    # ---- ONE SEARCH ----
    search_q = build_search_query_simple(org_min, query)
    internet_text = tavily_search_blob(search_q, max_chars=2000)

    # ---- POST ----
    post_payload = {
        "query": query,
        "org": org_min,
        "card_text": card_text,
        "internet_text": internet_text
    }
    post_raw = llm_json(SIMPLE_POST_PROMPT, post_payload, repair=1)
    post = post_validate_post(post_raw, query=query)

    out.update({
        "label": post["label"],
        "used_search": True,
        "search_query": search_q,
        "internet_text": internet_text,
        "rationale": post["rationale"],
        "evidence": post["evidence"],
    })
    return out


In [ ]:
import re

def post_validate_pre(out: dict) -> dict:
    """
    Нормализуем ответ PRE-агента.
    Правило: если label=1, evidence обязателен, иначе откат в 0 и needs_search=true.
    """
    label = 1 if int(out.get("label", 0) or 0) == 1 else 0
    needs_search = bool(out.get("needs_search", False))
    rationale = str(out.get("rationale", "") or "").strip()[:400]

    evidence = out.get("evidence", [])
    if not isinstance(evidence, list):
        evidence = []
    evidence = [str(x).strip() for x in evidence if str(x).strip()][:6]

    if label == 1 and not evidence:
        label = 0
        needs_search = True

    return {
        "label": label,
        "needs_search": needs_search,
        "rationale": rationale,
        "evidence": evidence,
    }


def post_validate_post(out: dict, query: str) -> dict:
    """
    Нормализуем ответ POST-агента.
    Правило: label=1 только если есть evidence.
    (Доп. эвристика: если evidence вообще не содержит токенов из query — откат в 0)
    """
    label = 1 if int(out.get("label", 0) or 0) == 1 else 0
    rationale = str(out.get("rationale", "") or "").strip()[:500]

    evidence = out.get("evidence", [])
    if not isinstance(evidence, list):
        evidence = []
    evidence = [str(x).strip() for x in evidence if str(x).strip()][:8]

    return {
        "label": label,
        "rationale": rationale,
        "evidence": evidence,
    }


In [ ]:
import random

n = 200
# берём случайные индексы, но пропускаем relevance == 0.1
random.seed(228)
idxs = list(range(len(train_data)))
random.shuffle(idxs)

sample_idxs = []
for idx in idxs:
    rel = train_data.iloc[idx].get("relevance", None)
    try:
        if float(rel) == 0.1:
            continue
    except Exception:
        pass
    sample_idxs.append(idx)
    if len(sample_idxs) == n:
        break

assert len(sample_idxs) == n, f"Не смог набрать {n} примеров без relevance=0.1"


# Метрики
TP = 0
FP = 0
TN = 0
FN = 0

for idx in sample_idxs:
    row = train_data.iloc[idx].to_dict()
    pred = simple_agent_predict(row)
    true = int(row.get("relevance", 0.0))

    pred_label = int(pred["label"])

    if true == 1 and pred_label == 1:
        TP += 1
    elif true == 0 and pred_label == 1:
        FP += 1
    elif true == 0 and pred_label == 0:
        TN += 1
    elif true == 1 and pred_label == 0:
        FN += 1

    print("="*110)
    print("idx:", idx)
    print("relevance:", row.get("relevance", None))
    print("Text:", row.get("Text",""))
    print("true:", true, "| pred:", pred_label, "| used_search:", pred["used_search"])
    if pred["used_search"]:
        print("search_query:", pred["search_query"])
    print("evidence:", pred["evidence"])
    print("rationale:", pred["rationale"])


# Метрики
accuracy = (TP + TN) / n

precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

print("\n" + "="*110)
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")
print("="*110)


idx: 12185
relevance: 0.0
Text: флюорография цена кузьминки
true: 0 | pred: 1 | used_search: True
search_query: флюорография цена кузьминки Medline-Service Москва, Грайвороновская улица, 6, стр. 1 флюорография цена кузьминки
evidence: ['Флюорография в Медицинском центре МедлайН-Сервис, метро Текстильщики, цены от 1600р., тел. - +7 (499) 495-22-38, мы работаем: пн-сат: 08:00 - 21:00,']
rationale: Медлайн-Сервис предоставляет флюорографию по цене от 1600 рублей, что соответствует запросу о цене флюорографии в Кузьминках.
idx: 1172
relevance: 1.0
Text: Ремонт глушителей
true: 1 | pred: 1 | used_search: False
evidence: ['замена глушителя', 'ремонт выхлопных систем автомобилей, включая замену гофр, глушителей и катализаторов']
rationale: Организация предоставляет услуги по ремонту и замене глушителей, что соответствует запросу.


KeyboardInterrupt: 

### Валидация на нормально размеченном датасете

In [ ]:
import random
import json, time, math, re
from pathlib import Path

# === CONFIG ===
JSONL_PATH = "data_final_for_dls_eval_new.jsonl"  # put correct path if different
N_ITERS = 200

# If your label field names differ, update these:
TRUE_LABEL_KEYS = ["relevance_new"]
PRED_LABEL_KEY = "label"  # from simple_agent_predict output

def safe_float(x):
    try:
        return float(x)
    except Exception:
        return None

def get_true_label(obj: dict) -> int:
    """
    Try common keys. If value is float like 1.0/0.0/0.1 => map to 1 if >=0.9 else 0.
    If value is int 0/1 => keep.
    """
    for k in TRUE_LABEL_KEYS:
        if k in obj:
            v = obj.get(k)
            fv = safe_float(v)
            if fv is None:
                continue
            return 1 if fv >= 0.9 else 0
    # fallback
    return 0

def get_row_for_agent(obj: dict) -> dict:
    """
    Convert jsonl record into the dict expected by your simple_agent_predict:
    must contain at least 'Text', plus org fields like name/address/rubric if present.
    """
    row = dict(obj)
    # normalize query field name
    if "Text" not in row:
        # common alternatives
        for alt in ["text", "query", "request", "user_query"]:
            if alt in row:
                row["Text"] = row.get(alt, "")
                break
    return row

def metrics_from_counts(tp, fp, tn, fn):
    total = tp + fp + tn + fn
    acc = (tp + tn) / total if total else 0.0
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * prec * rec) / (prec + rec) if (prec + rec) else 0.0
    return acc, prec, rec, f1

# === LOAD JSONL ===
path = Path(JSONL_PATH)
assert path.exists(), f"File not found: {path.resolve()}"

records = []
with path.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        records.append(json.loads(line))
random.shuffle(records)
assert len(records) > 0, "JSONL file is empty"

# === EVAL LOOP ===
tp = fp = tn = fn = 0
logs = []

# Evaluate first N_ITERS records (or fewer if file smaller)
N = min(N_ITERS, len(records))
skipped = 0
for i in range(N):
    obj = records[i]
    row = get_row_for_agent(obj)

    # run your pipeline (must exist in notebook)
    pred_out = simple_agent_predict(row)
    pred = int(pred_out.get(PRED_LABEL_KEY, 0) or 0)
    pred = 1 if pred == 1 else 0

    true = obj['relevance_new']
    if true not in (0, 1):
        skipped+=1
        continue

    if true == 1 and pred == 1:
        tp += 1
    elif true == 0 and pred == 1:
        fp += 1
    elif true == 0 and pred == 0:
        tn += 1
    else:
        fn += 1

    acc, prec, rec, f1 = metrics_from_counts(tp, fp, tn, fn)

    log_item = {
        "i": i-skipped,
        "true": true,
        "pred": pred,
        "text": row.get("Text", ""),
        "used_search": pred_out.get("used_search"),
        "search_query": pred_out.get("search_query"),
        "evidence": pred_out.get("evidence"),
        "rationale": pred_out.get("rationale"),
        "counts": {"tp": tp, "fp": fp, "tn": tn, "fn": fn},
        "metrics": {"acc": acc, "precision": prec, "recall": rec, "f1": f1},
    }
    logs.append(log_item)

    print(
        f"[{i+1:03d}/{N}] "
        f"acc={acc:.4f} precision={prec:.4f} recall={rec:.4f} f1={f1:.4f} "
        f"| TP={tp} FP={fp} TN={tn} FN={fn}"
    )

print("\nDone.")
print(f"Final metrics on {N} iters: acc={acc:.4f} precision={prec:.4f} recall={rec:.4f} f1={f1:.4f}")
print("Logs saved to `logs` list.")


[001/200] acc=1.0000 precision=1.0000 recall=1.0000 f1=1.0000 | TP=1 FP=0 TN=0 FN=0
[003/200] acc=0.5000 precision=1.0000 recall=0.5000 f1=0.6667 | TP=1 FP=0 TN=0 FN=1
[004/200] acc=0.6667 precision=1.0000 recall=0.6667 f1=0.8000 | TP=2 FP=0 TN=0 FN=1
[005/200] acc=0.7500 precision=1.0000 recall=0.7500 f1=0.8571 | TP=3 FP=0 TN=0 FN=1
[006/200] acc=0.8000 precision=1.0000 recall=0.8000 f1=0.8889 | TP=4 FP=0 TN=0 FN=1
[007/200] acc=0.6667 precision=0.8000 recall=0.8000 f1=0.8000 | TP=4 FP=1 TN=0 FN=1
[008/200] acc=0.7143 precision=0.8333 recall=0.8333 f1=0.8333 | TP=5 FP=1 TN=0 FN=1
[009/200] acc=0.6250 precision=0.8333 recall=0.7143 f1=0.7692 | TP=5 FP=1 TN=0 FN=2
[010/200] acc=0.6667 precision=0.8571 recall=0.7500 f1=0.8000 | TP=6 FP=1 TN=0 FN=2
[013/200] acc=0.7000 precision=0.8571 recall=0.7500 f1=0.8000 | TP=6 FP=1 TN=1 FN=2
[014/200] acc=0.7273 precision=0.8750 recall=0.7778 f1=0.8235 | TP=7 FP=1 TN=1 FN=2
[015/200] acc=0.7500 precision=0.8889 recall=0.8000 f1=0.8421 | TP=8 FP=1 TN

## Сохраняем логи в txt

In [ ]:
with open("logs_list.txt", "w", encoding="utf-8") as f:
    for item in logs:
        f.write(f"{item}\n")


In [ ]:
with open("logs_list.txt", "r", encoding="utf-8") as f:
    logs = f.readlines()

In [ ]:
import ast

logs = []
with open("logs_list.txt", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        d = ast.literal_eval(line)  # строка -> dict
        logs.append(d)


In [ ]:
logs

[{'i': 0,
  'true': 1.0,
  'pred': 1,
  'text': 'стоматологии, стоматологические клиники',
  'used_search': False,
  'search_query': '',
  'evidence': ['Рубрика: Стоматологическая клиника',
   'Услуги/цены (summary): Стоматологическая клиника «Фамилия» предлагает широкий спектр услуг: от пломбирования и удаления зубов до имплантации и протезирования, а также профессиональную гигиену и отбеливание зубов'],
  'rationale': 'Организация явно указана как стоматологическая клиника и предоставляет стоматологические услуги.',
  'counts': {'tp': 1, 'fp': 0, 'tn': 0, 'fn': 0},
  'metrics': {'acc': 1.0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0}},
 {'i': 1,
  'true': 1.0,
  'pred': 0,
  'text': 'лучшие рыбные рестораны москвы рейтинг 2016',
  'used_search': True,
  'search_query': 'лучшие рыбные рестораны москвы рейтинг 2016 Рыбная мануфактура № 1 Москва, улица Ефремова, 10с1к4/2 лучшие рыбные рестораны москвы рейтинг 2016',
  'evidence': [],
  'rationale': 'Запрос относится к рейтингу 2016 года